CHECK DATABASE INTEGRITY AND TABLES

In [24]:
import sqlite3
import pandas as pd
import os
from pathlib import Path
# current_dir = Path(__file__).resolve().parent #not working in jupyter notebook
current_dir = Path(os.getcwd())

parent_dir = current_dir.parent
db_path = parent_dir / "data" / "Data Engineer_ETL Assignment.db"

def run_command(query):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(query)
        results = cursor.fetchall()
        conn.close()
        return results
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        return None

run_command("SELECT 1")
# print(tables)

An error occurred: database disk image is malformed


TRY : Database Recovery, downloading sqlite3.exe to current directory(tests) and run recovery

In [21]:
import sqlite3
import os
import logging

SOURCE_DB = db_path
RECOVERED_DB = parent_dir / "data" / "recovered_database.db"
DUMP_FILE = parent_dir / "data" / "db_dump.sql"


logging.basicConfig(level=logging.INFO,format="%(asctime)s - %(levelname)s - %(message)s")


def check_integrity(db_path):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        cursor.execute("PRAGMA integrity_check;")
        result = cursor.fetchall()
        conn.close()

        if result[0][0] == "ok":
            logging.info("Integrity OK.")
            return True
        else:
            logging.warning("Database corruption detected.")
            return False

    except Exception as e:
        logging.error(f"Integrity check failed: {e}")
        return False


def dump_database(db_path, dump_path):
    try:
        conn = sqlite3.connect(db_path)

        with open(dump_path, "w", encoding="utf-8") as f:
            for line in conn.iterdump():
                f.write(f"{line}\n")

        conn.close()

        logging.info("Database dump created successfully.")
        return True

    except Exception as e:
        logging.error(f"Database dump failed: {e}")
        return False


def rebuild_database(dump_path, new_db_path):
    try:
        if os.path.exists(new_db_path):
            os.remove(new_db_path)

        conn = sqlite3.connect(new_db_path)

        with open(dump_path, "r", encoding="utf-8") as f:
            sql_script = f.read()

        conn.executescript(sql_script)
        conn.commit()
        conn.close()

        logging.info("Recovered database created successfully.")

    except Exception as e:
        logging.error(f"Database rebuild failed: {e}")


def main():
    logging.info("Starting SQLite recovery process")

    integrity_ok = check_integrity(SOURCE_DB)

    if integrity_ok:
        logging.info("Database not corrupted. Recovery not required.")
        return

    dump_success = dump_database(SOURCE_DB, DUMP_FILE)

    if dump_success:
        rebuild_database(DUMP_FILE, RECOVERED_DB)
    else:
        logging.error("Recovery failed. Dump could not be created.")

main()

2026-03-14 19:21:06,386 - INFO - Starting SQLite recovery process
2026-03-14 19:21:06,388 - ERROR - Integrity check failed: database disk image is malformed
2026-03-14 19:21:06,390 - ERROR - Database dump failed: database disk image is malformed
2026-03-14 19:21:06,391 - ERROR - Recovery failed. Dump could not be created.


iterdump() failed - trying .recover method

In [25]:
import subprocess

SOURCE_DB = db_path
RECOVERED_DB = parent_dir / "data" / "recovered_database.db"

def recover_with_cli(source_db, recovered_db):

    try:
        cmd = f'sqlite3 "{source_db}" ".recover" | sqlite3 "{recovered_db}"'

        logging.info("Running CLI recovery.")
        subprocess.run(cmd, shell=True, check=True)

        logging.info("Recovery completed successfully.")
        return True

    except subprocess.CalledProcessError as e:
        logging.error(f"CLI recovery failed: {e}")
        return False


if not os.path.exists("data"):
    os.makedirs("data")

success = recover_with_cli(SOURCE_DB, RECOVERED_DB)

if success:
    logging.info(f"Recovered database saved at: {RECOVERED_DB}")
else:
    logging.error("Recovery process failed.")

#CHECK : integrity of recovered database
if check_integrity(RECOVERED_DB):
    logging.info("Recovered database integrity verified.")
else:    logging.error("Recovered database integrity check failed.")

#CHECK : query db
print(run_command("SELECT name FROM sqlite_master WHERE type='table';"))



2026-03-14 19:31:41,584 - INFO - Running CLI recovery.
2026-03-14 19:31:41,625 - INFO - Recovery completed successfully.
2026-03-14 19:31:41,626 - INFO - Recovered database saved at: c:\Users\admin\Documents\eastvantage\data\recovered_database.db
2026-03-14 19:31:41,628 - INFO - Integrity OK.
2026-03-14 19:31:41,629 - INFO - Recovered database integrity verified.


An error occurred: database disk image is malformed
None


RECOVERY : Failed, will try to rebuild based on given schema.